# Your First Metasurface in 5 Minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/metasurface-py/blob/main/notebooks/01_getting_started.ipynb)

This notebook demonstrates the core metasurface-py workflow:
1. Define a rectangular metasurface lattice
2. Apply a beam-steering phase profile
3. Compute and plot the far-field radiation pattern
4. Compare continuous vs quantized phase states

In [ ]:
# Install metasurface-py (uncomment for Colab)
# !pip install metasurface-py

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from metasurface_py.geometry import RectangularLattice
from metasurface_py.elements import PhaseOnlyCell, DiscretePhaseSpace
from metasurface_py.elements.states import ContinuousPhaseSpace
from metasurface_py.surfaces import Metasurface
from metasurface_py.em import steering_phase, far_field_pattern, peak_gain_db
from metasurface_py.core.types import AngleGrid
from metasurface_py.plotting import (
    set_publication_style, plot_pattern_comparison,
    plot_state_map, plot_lattice
)

set_publication_style()

## Step 1: Define the Metasurface

We create a 16×16 rectangular lattice at 28 GHz with half-wavelength spacing.
This is a typical configuration for 5G mmWave RIS research.

In [ ]:
freq = 28e9  # 28 GHz
lattice = RectangularLattice.from_wavelength(
    nx=16, ny=16, spacing_fraction=0.5, freq=freq
)
print(f"Elements: {lattice.num_elements}")
print(f"Spacing: {lattice.dx*1e3:.2f} mm")
print(f"Aperture: {lattice.extent[0]*1e3:.1f} x {lattice.extent[1]*1e3:.1f} mm")

fig, ax = plt.subplots()
plot_lattice(lattice, ax=ax)
ax.set_title("16×16 Element Positions")
plt.show()

## Step 2: Steer the Beam

We compute the progressive phase shift needed to steer the main beam to θ = 30° in the φ = 0° plane. The steering phase follows from the array manifold:

$$\phi_n = -k_0 (x_n \sin\theta_s \cos\phi_s + y_n \sin\theta_s \sin\phi_s)$$

In [ ]:
theta_steer = np.radians(30)
phase = steering_phase(lattice, theta_steer=theta_steer, phi_steer=0.0, freq=freq)

# Create surfaces with different quantization levels
cell_cont = PhaseOnlyCell(state_space=ContinuousPhaseSpace())
cell_2bit = PhaseOnlyCell(state_space=DiscretePhaseSpace(num_bits=2))
cell_1bit = PhaseOnlyCell(state_space=DiscretePhaseSpace(num_bits=1))

surface_cont = Metasurface(lattice=lattice, cell=cell_cont)
surface_2bit = Metasurface(lattice=lattice, cell=cell_2bit)
surface_1bit = Metasurface(lattice=lattice, cell=cell_1bit)

state_cont = surface_cont.set_state(phase)
state_2bit = surface_2bit.set_state(phase).quantize()
state_1bit = surface_1bit.set_state(phase).quantize()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, state, title in zip(axes, [state_cont, state_2bit, state_1bit],
                             ["Continuous", "2-bit", "1-bit"]):
    plot_state_map(state, nx=16, ny=16, ax=ax)
    ax.set_title(title)
fig.suptitle("Phase Distribution — Steering to 30°", y=1.02)
fig.tight_layout()
plt.show()

## Step 3: Compute and Compare Far-Field Patterns

Phase quantization introduces gain loss. Theoretical values:
- 1-bit: ~3.92 dB loss
- 2-bit: ~0.91 dB loss

In [ ]:
angles = AngleGrid(
    theta=np.linspace(0.01, np.pi - 0.01, 360),
    phi=np.array([0.0]),
)

patterns = []
for surface, state, label in [
    (surface_cont, state_cont, "Continuous"),
    (surface_2bit, state_2bit, "2-bit"),
    (surface_1bit, state_1bit, "1-bit"),
]:
    pattern = far_field_pattern(surface, state, freq=freq, angles=angles)
    gain = peak_gain_db(pattern)
    print(f"{label}: Peak gain = {gain:.1f} dBi")
    patterns.append((pattern, label))

fig, ax = plt.subplots()
plot_pattern_comparison(patterns, cut_phi=0.0, ax=ax)
ax.set_xlim([0, 90])
ax.set_ylim([-30, 0])
ax.set_title("Beam Steering to 30° — Quantization Comparison")
plt.show()

## Summary

In this notebook we:
- Created a 16×16 metasurface at 28 GHz
- Applied analytical beam-steering phases
- Compared continuous, 2-bit, and 1-bit quantized patterns
- Verified quantization loss matches theory

**Next:** See [02_optimization.ipynb](./02_optimization.ipynb) for hardware-constrained optimization.